# Continue downstream-aware FastViT fine-tuning

This notebook resumes from the selected checkpoint produced by the first continuation run. It performs at most six lower-learning-rate last-stage epochs and stops at the first accepted checkpoint. The revised objective matches WHAM pose in angle space and adds low-weight ground-truth pose supervision from the official 3DPW train split. WHAM stays frozen. It still selects only on validation and never opens the 3DPW test split.

Before running, save the epoch-24 continuation notebook as a Kaggle version and attach only that notebook output as the phase-three checkpoint input. Also attach the same `3dpw-model` and `3dpw-vit` datasets. Only the selected checkpoint, history, and report remain in `/kaggle/working`.


In [ ]:
# Configuration. Change a mount if Kaggle assigned a different dataset slug.
from pathlib import Path

CONTINUE_FROM_PHASE3 = True
POLISH_PHASE3 = False
PHASE2_NOTEBOOK_ROOT = Path(
    '/kaggle/input/notebooks/nguyntrunglong/'
    'distill-fastvit-hmr2-kaggle7f1f81131a'
)
THREEDPW_ROOT = Path(
    '/kaggle/input/datasets/nguyntrunglong/3dpw-model'
)
PARSED_3DPW_ROOT = Path(
    '/kaggle/input/datasets/nguyntrunglong/3dpw-vit'
)
OUTPUT_DIR = Path('/kaggle/working/fastvit_hmr2_phase3')
SCRATCH_DIR = Path('/tmp/fastvit_hmr2_phase3')
if CONTINUE_FROM_PHASE3:
    phase3_matches = sorted(
        Path('/kaggle/input/notebooks/nguyntrunglong').glob(
            '*/fastvit_hmr2_phase3/fastvit_hmr2_best.pth'
        )
    )
    if len(phase3_matches) != 1:
        raise FileNotFoundError(
            'Attach exactly one saved phase-three notebook output; '
            f'found {phase3_matches}'
        )
    SOURCE_CHECKPOINT = phase3_matches[0]
else:
    SOURCE_CHECKPOINT = (
        PHASE2_NOTEBOOK_ROOT / 'fastvit_hmr2_phase2/fastvit_hmr2_best.pth'
    )

CLIP_LENGTH = 32
STRIDE = 16
BATCH_SIZE = 2
WORKERS = 4
HEAD_EPOCHS = 0 if CONTINUE_FROM_PHASE3 else 2
LAST_STAGE_EPOCHS = 10 if POLISH_PHASE3 else (6 if CONTINUE_FROM_PHASE3 else 4)
FINETUNE_HEAD_LR = 5e-6 if POLISH_PHASE3 else (1e-5 if CONTINUE_FROM_PHASE3 else 2e-5)
FINETUNE_BACKBONE_LR = 5e-7 if POLISH_PHASE3 else (1e-6 if CONTINUE_FROM_PHASE3 else 2e-6)
ROTATION_LOSS = 'geodesic' if CONTINUE_FROM_PHASE3 else 'cosine'
WHAM_POSE_WEIGHT = 4.0 if CONTINUE_FROM_PHASE3 else 25.0
WHAM_ROOT_WEIGHT = 1.5 if CONTINUE_FROM_PHASE3 else 10.0
WHAM_GT_POSE_WEIGHT = 1.0 if CONTINUE_FROM_PHASE3 else 0.0
WHAM_GT_ROOT_WEIGHT = 0.0
MAX_CLIPS = 0  # 0 means every available training clip.

if not SOURCE_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f'The configured source checkpoint is missing: {SOURCE_CHECKPOINT}. '
        'Attach the corresponding saved notebook version.'
    )
if not THREEDPW_ROOT.is_dir():
    raise FileNotFoundError(
        f'Change THREEDPW_ROOT to your registered private 3DPW input: {THREEDPW_ROOT}'
    )
if not PARSED_3DPW_ROOT.is_dir():
    raise FileNotFoundError(
        'Attach your existing 3dpw-vit dataset and set '
        f'PARSED_3DPW_ROOT to its mount: {PARSED_3DPW_ROOT}'
    )
sequence_candidates = (
    THREEDPW_ROOT / 'sequenceFiles/train',
    THREEDPW_ROOT / 'sequenceFiles/sequenceFiles/train',
    THREEDPW_ROOT / '3DPW/sequenceFiles/train',
)
SEQUENCE_TRAIN_ROOT = next(
    (path for path in sequence_candidates if path.is_dir() and next(path.glob('*.pkl'), None)),
    None,
)
if SEQUENCE_TRAIN_ROOT is None:
    raise FileNotFoundError(
        'The attached 3dpw-model input has no sequenceFiles/.../train/*.pkl. '
        'Expand the nested sequenceFiles tree and verify that train exists. '
        'It is required to map WHAM numeric training '
        'track ids to the correct image sequences.'
    )
print(f'Phase-two source: {SOURCE_CHECKPOINT}')
print(f'3DPW root: {THREEDPW_ROOT}')
print(f'Parsed 3DPW tensors: {PARSED_3DPW_ROOT}')
print(f'3DPW train annotations: {SEQUENCE_TRAIN_ROOT}')
print(f'Final artifacts only: {OUTPUT_DIR}')


In [ ]:
# Kaggle supplies CUDA PyTorch; do not replace it.
%pip install -q timm==1.0.22 einops==0.8.1 yacs==0.1.8 joblib==1.5.2 loguru==0.7.3 smplx==0.1.28 opencv-python-headless==4.12.0.88 scikit-image==0.25.2 tqdm==4.67.1


In [ ]:
# Materialize the reviewed scripts in temporary storage and verify their bytes.
import base64, gzip, hashlib

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
embedded = {
    'evaluate_wham_feature_substitution.py': ('0a9c5fe4039b32c2b7c070f0e2a22dd496b88a23f238ddf5dbbce2b5ae3a0730', 'H4sIADdgoWoC/809XXPbOJLv+hUc7sNRCUXLcpzJaEdXm80kO7M1mUtlsjsPPhWLpiCbY4pkSMq24vX99usPAARASrYne1U3tRVTQKPRaKAb/UXun7452jb10XlWHIni2qt27WVZnIx833999NdJK5rWK9frLM2S3Pvx/ceZlxQrb5U1bZbnYuW9S5r2n9knby2SdluLxsuKJlsJ77cfX7+PRqNPl6IbXiV1A0NOfvjwm7fOcuE126rKMxi0Eq1I26wsmpAnUejkz0Js6yQfZUXWAqLsS4KwIZFyUZfbYjVp62176f36/sPPXl221N9E3sfkxqvFBVArajVztkkuYMqkFiNaBAB83mbY3ZZeWm6qbSv6yyoLr4W1iNskbb0m2QhvXcO/Da0xw2W3osBZkzzfeeI6ybcJMI8G1SLd1jV0E1dgjlp4NxnwedsyxUnTiBbI/akd1aIq67bRi5g0VZIKYHhyUZRAbwosKcqW8FZJJer/aLz3H/7+4e3Rh3++9TYiaYDcDcwFlMEejkbrutx4cbze4jriGJaPEwDrCsWm0Ui11Re0Rep32lyrx8ukucyzc/WT/0BDtAUWqtbfm7JQz832vKrLVDSNbtnpxzbbCCYsLeEU8c5HyXmqqHsDXEzOcwlUJS1Orjo/wE/uaHdVVlyo9tfFTi/l9/LcILfYbqodcNkrKoOEjX4u6/TS+hEVRbTeFilvKI58xzN++OlnNd1PeI4kHThGtReFbPy82qg2fB6NXn988+NPn96++fSPj2+9heev4ZBdZ23cJLMXMewznu34clPP4uuZP8KzEr/5r/fvf/qEwLPz0xfrb7/97tuT747T7168+vblqxevzr+bnorVq29Pz49PX6Qvktl3p7Dlo5VYA00xLTvAoyjmyJ2xN/lPYEFUrJK6TnbzkQf/ZWsva0Bo26RIBQOHkgmfRNGU9Zjh8L9awCEqPAKKQGaT9DIYR2m1hX95svHIgIOpkoamYrxjSVpzmcxOX8aoAgLc2zltKVHXtDVPt8ouUPUs1MmLeJCcAKWHjkVUVqII/PrcH+MuwXCRbDqC12XtpZfb4grk08tACQR5sjlfJXMJGcE/q+CV98w7ns5eyD/j0Dv3fWPZHT3RtlqBWAeE01qr7L8Ut/wUqMWCiKaoGXLJ3GZubEHofZ5767xMWlo9Pc1NtNQSwIAeGmAtHH+BfQT08kUI0lTtFu+SvBGwhs9jze/tZpPU2ZchCmjeVZa2Z8CRkOfz/oXqbMmErPMEt+FRkwI7YZugf3JsMedOs9IH1VnlovHnOEWAyKMGKBuHHQgoscKXbGEIbAnGJkx1OgUQgykIF3qnUwsIpGEA6LtTa7bk1pksudVz3UsOJrdZEyfFRS5ikKtN0tbZbdA1zm2BQZaaDcxIgmyAldyVZ6BZLqJr0H5lHRdlvTEQhrAlm8XkOPSuhKjw+VON8oN4LpN8HWtk8uGZN41OqbvZgO7UHaBUm2Dsfe8di8lL7k8TuHkVFWJTtbs4z65EwAPGHdDZ/xCupQYGNREYs6v+sXfk2S0GDo0C6PMmCo5bo+bzFq7hABG8eBVNadhnvDfrAhSvnjeFrQnkY9mYJMAx75gGTKA5x4p76gwmyI8O71kURaE3P2Yy0RyArah3AzDHc4Zpb8oYmT2LpkBqB6UXEIGI6UMP17qoAVpjjrYFAArxhQQDyBzsmfHotC6b7pR8EfCT94fQMsxt6IHq+OLMASbcShNBaHgVU1gI7sDky0DPDHt2bgccvCl2fBnooBGTW7dnJodMdgM9ND8PAeMQhLHddUdwJ4IT2DJULItuNyNqgHZxnaV2B7VYCkYjfc57xRP/UhaC/10C0wMt87xBE4N57j4OjVdzaDTP6Wh1oAhEyyYIpXxZWaDaUCZd/HIVcOuj9IZcIo+QZxdYOl9GaQ6zBp3WfcYwEf06m09my9B7qegwZjd0mGp9FCXrrG5AfzYiLcHwXmiUkqgTmM5uOpHyQwNhwLsI9Rwa8KCLGZkpqRqxCafZLXsnHg9FaacWtWmOroR+nmLUXeGkUuXukFxfZvWqUzO4d4G1SFeT0E5IZQjWz1XgghNGpYBmivcXolyJJkvjlbiohWhiNBB5C8BC5iVW4IFkaX8vACfY5aK1W0d7DwtcYdk1KvgOofcXiSNq66RoqrIRxC+tckrQ7DgkCNT4CB0OtH0DWMkxLIWWNENWKBmBnTiOpqi7QS2CFQiWVdXt1gT6QgQw2G3xr05WM2CH1OsJKnamQ5stSGcs6hqux+QiQRM17rSAyzQ41sN8i/dd0/tYaPBtsUdqrGkNowd03QsUOXm6aH68ioeshx51PTwnY4ttDxyijia1couPqzpbt3xaB1jFx7fXcUgP7GWOmmMvX3r6wx6vadmL4HEMsSQTeQEnFG5W6dljWOHXdos6PQBn73252oLlIH0PYFocY7AhjoGcfE18QBXf+QTNFuxK0L0abmwoqnwdnYN2OC9JqtDVBOUiwGqIN0ByHlieheUG+iGePxBTEIQVW9Uhuq8xES+axVSP7SZML8GbFzlaDdbc6JPFKoJhk6e9TfB/YBhw4FcBhkiBjTZ50PWmLK5nq0BNA2I9e4XatoZfMVrvC9ig8yxplO/hIvhbXW6rX9DEPX5JowdA3v78j6Df/HqVVKiPXl9ffCjLHKgIWDJ6kO9AcbXgC/Z7fk52oubZZ+jqoZ93MgAGLE9qEyaU7uAAy4mJKrIUn2/XazgNvpRpcmBC04ILCNEjhzftSo+GXdSD9dkEr/YmqVd0NEOOZz1Kbkl2VUDLPSsSqT4vAeE9m8PFTv87mc0nJ7NltwZ9Ra8ULvNQBQrP2I0eGOOe8bhu3WBTmS3ISKnEwD9bAQQLLN9alyK9qkpwI+MuiKDsRcUP/iVV/RY8z7O+/IeG//u62C2l49vh73w2ICJw5g3BMqvivExJlS38tNrC7t2I7OKybeKyyJVzzE4goMkw1gmsAbQdrgjWG/hmtz9W8Rlr0DcLz4wjGcGZJGuE93FbYHTtLV6btiCv/be3FSABvt+ZGL6p78HvxyCqd2fOBO0YNblz1nvvO+JAKg2Ntr5iHUcYCA0MuEjuI+pC5LrBzTNfbq/R7YNJ2aA+bw0HmLRjVlzEHNdAT8IIMFgsnFu8M7x+UZXpJXS7G8DtZnwAiLkQA5DcbkImaSoqYO8AsO7qw2PYjVe8d5wBYo4H1mYrOnUDI41OKyRymTTiJD44tA8zgGEj2gR6k/3jNYSOpZjeDB2GtgykMxf2dtWU+5vLBG7AspZWH/3EQLmS+SerAn3js/g0uyaiaCIYmaJugymdukDPIzU3hXUxFknUNxH2qwjvLwIcz/rK8/7kVbscKJljkgRj2AuGmIDXiZmRSVuCGroWOevzQo5bKAyGw7Op8gX5oLpJmnKLaXRs+DWx2JyDtXB6POsaizjHa69ZnJiAqJQXeJ10jXVRxOR6+z//+um9L90jU3D/TxQhHWdLcq/Ebs5hRit8C80hN6M2MtUF7YK/jEC8N41hf4HGxAwJDEQnrW4bjBaDwAI7I6lU+SxusgYcjovQ2xZCqcaF2pGepqJHrY6MpcB8EpMH9Ha4nqKaOTHUMRoQgiGbwlGWqBd38uHeJHdx1z33FLO6b+V6OmEzRUsmaWK67QNHskhWVC6GXXq+RM0Idvesbk48DcBIjQxcRB/E5ojF5ggMXJz1SJkeR5hDAgHc8QIaWA5FtswUU4StbOBj2kAfs8CXyqEuq5jwoAWNpKutIXRZQ+Y77g/+pr0VtWo+uFFr/025zVd0pKSs82bBjB7OmLVwbu/4atQ33ZZirM4auJ1XgWQE+M9YL1pSFYlbWCrDBvxn7KhOaIqsjdM7muJxpaa4Lss2wH+MvcQHad0kxQrVO1mC3WlEeNwuQvEuw0i98zM8BNvvxJSr/1SMQ4NsKDOKQ0ketRxSEnptc1cp6K4I3O5VBh6cneYBX7HNCkcFNeQapagHags/qp56AAseO2OMnosS1iCwrS3+7BEY8Bd5eR7w4uNn0e/VBdyhdFKtYWM8v7go+ww75rYmdtQdcOToL2X7Do0+Rx0Zp32dAbWJkTSnjYBrraacxY6zcIpw4FQOp7fxOjW09pGdfK17xmq8c5GXN94dbqTUWjpVpdjA4ChVMj7WHWp1vV/DbVrG2YqSeSHn4+Gnld7i6z4HP+sMB0kFpWcpMJG/0JiiuqlAnkGpgBY5Hp9NlyN3b2TMnSmBI2qhGhknbfAAuJqmvxFr/728S4jvmrua66BrTNRK54CSjmVlA3hjoHUE+oeHz9RYa2vgWccjQHC21JKl2IoHX7PYyAPCbfXyhbEwJpRcQoNrR3AYeO47yvtJROP59HR1T9SYkso4kGuUI3akiwiOkqoSxSpg0O7yFzmMn3rfLzxrHu97LxdF0HHpEM4O6sxCsjSnaRyRe1CyOqn4pWROejYv7onfd+owsvdl77Ut/+atQAuQQtR52Hw5ZkW1lSoHjCIyMJws9Pl5eWu3gOde5lsOR7vidNAGkNcL+HGUBEPMFESYLd3k8YkMP1O9UAc5m5/sBU3LEvRtIa8tTMc98wK9JpUcWXoTSQDi4yzOnNKUOJXZJosJVqhOLslSBbScC+4YAGpgHPZbj+VxyMtClipg5thCxgDGdmi2MOny55HGMTFrJvRmnxHSDkzTqluWKn/WMcy4IJWxhMZtFYFORisBmRgEPdpC3g6NWWZYdQ7kdgCJsSs6ZovSpjcGkJxgWYUixEW6SRr0gJyNhH383ptGJ+Yxvx04GSGN12muspVLAY8xLYPfe8d9sAyGwYCI382jdPwt5bX6k8qyhy4oUYn8OnPHnx1j7vN4Sdk6KmLgZc90rqoxUMglSkomEiWs6y/sdGXFWtSkC9CQBocH1wvGRJ2krQqNdeE76WVR63wgOBMat2rvCuDOnlaQ+RHD8Jw/yT2QaNGxobgxFYGEMq7Z99VDU93YCQvaBqlrwLEG/aboN+GMq0zkSdWQfzeVOSmy7NA/RD2LBVqGFZzA2ccoAJ5ig0VYGaSJN4IiK9GkC9+tGTRM1lwk10KG842bkuMDZC3sGnBWwX5CWyFp252KYxv3VEtremCdamXskcMG4uq+ZJV9DxlrOmMezCUvnhsrXNoxclLQj4J2LleyE6lcjsu1yD+jcq1yW6dDpuvFOasZpQoZELXONcZn/I9/+yuaxFrnbeEcvRrbEX2t2UIvpnSo6egOTRn2GulGmi37HXwZYPeM7pXZFE5VH2x2+vIRjTbdcpeVLdIbzvtOPiSX3BlsMtfs3gfj8Z5JtdVo5rUlFeMuaoBlp0V8Dir8CsMRndqS0ZyaoyfoOGOx2hq07xa1sBGYMXMPMkIsjTHL+KPpIiTfWyw8P92uEt8+IzJfDx1RsyvSy7ossFTANM9Y2p8PEgS6VRJsloSgDlFcV5Qy81SN40AWG0uT5Fg4j3LaB1R2vS3ciKaM0cy7wCQfEadCJNS35VC7Inqoj5KUV9XeLgws7u1kp6vfmcIaQFNei9ztdaoKLV01N28RK+ImA0PQHNyq4ha+3Mc6Mw+LOIHTvSkpcYz+JdyABg7ZATwHhteBniZULDBQUSCBH68phaFDZXXyO3tb8UowIn1S7JnDjkOhwQ/D+MLC8Is6sWOLqhH8AxefnTcj4nBzJJ1kXMlnmE49oZudDrChR31HTdhtfL9Ew8inIABWUWpCui7cCtVF29J1IUNUFz6j0X08X5rZCwGqJGt3Cgp+m1kXXKfq4kV3nbBwUSeqF9lg9DErdCf/tKs5dd2BTDUEVGRyqCwW04LzPQxCb5FtNixEgIm7clvGO3b4sgcMWYSeRgd8Xq52B4CBn06Z6ibJisApVqCKfnQ5VHV/9Lq+2OIbAh+oJzDNhg3m0GsuNlj0Bvwg1sk2b5sfRV69U8DG4eGpomS1ihM5JPAnE37tY3Kyqm4wr43XEgdw1IsXphk9jKK9rIWYAIIJHaw/iEVeOpMuzv5HMaH6nmBo+6sQfD0dKjrQKAQZpbN5nxbH04OD+QWWwZEn0+mjOEnW3wStv0E0L18cxAL+8gRVygQLeBLONeKzwkWXr7GcaPogOlW/1sO7B+c0On4YaSsS2Kp6QjVUBwg8fYBANhT+6F6DDaP3ewDV4a1OsnwCzK0F3mswLEk5SdfAFSTiFuZWccT6Au0ziYf+ICbMr0n9x/RSZM7OHSAYj1jFKOyh3aM81O7UOwBsEx3sHcgmYTvmjsyMgErHgVeEeVz2N5RXhA6RswoZsuUUsAo6Lp3kXmeCSuaQJg50vFa/RkY4554P3hFw2o/wcggkEmlDSvNvs8koUqrflYpo+THvrhEA8i8y3DW/FtesTvHHj29f/4AFEunNamGzCI4F2BJ0pGQ6F7OWVaCTYub83yw841WjP1RPIl9rI3R3BrKusMSYsJenNOLpi4EUFi2NLoAYDhW1RRSGuxbKJGeXQbsw/DNgzwGXa/gLWG15DdKAnjdclRjH9ShVrewtCsn6P8hwxB3juv8zsOPGfH8Qujoa732MSGybSynFHHVLzrkmj99F45y5KyNjM0wTqodYFUMQQ4zCp32C5Jk18arfqLS/61e+uFOdyZ7lQO1LH5Z7TFhdv2GUwfTH6c7l4UKYfSMNoOVgKUwM+xRzITGgCfp4jLKWJeaD7+7HVLZi12R2WLiarzuyh4ppzLXbCPt09KtsJDndVHbtUAz4+zU599axlbE9o5IAzim+lhmttpuqCdyjMXZPrl2S4hTf2EqmE8JwWH9bh9IKvyjUdkxmH3opT1QPgXqY41/4hpqdqcqKlbgNUVBjlUNBXS+KLboLrQhYIuEIZHD+jBiVhl5Q2Y+JwIpMBFbmb3/ScLwnuaxWoKIMRPDYTBrqNT5aC2MyyXyvWicMMR/ZMEJ8tflSv229GkzmukqZ39PjmAmmNuAKY+WjLE4OjiqCVS4RRIZe5uXwXZ4V9PqyCqVqaHx3gOqaF+Y8RtCKs4oy+5qr0pwzhYDycmqy8ZLvdvkT91zTsRzp7Ki04uIuTHIojooj3Jj6wyO4bvUxgJIYjrs8lpCHocFIjNUWSUinuNWQGV5WzBXyjREaJyHsmu98SS6oNYxzeqpSk3/fj2wRxA1Qm7ZXyCxZPKNxRnpVZ4LoTpeBTjXkqpoNjcHw9BA4tg+A61z20BjVOTCuywAODcSwvxqDmqCzYZXRIYXJEmKSLHbH7NCwnUUjqen14/r2dOk17unXy1UZC73cw/AUE9oDa2nMbtXfw+l6qNSGqG1yNuXobwBaosPx3Dsej6yMvVlm0KsYsZMdHKJTp9AoFzkz5l0Olfo7WkC9UoKTH86+ORjCfsXBQILFpMYZYd6WYf8AKSK6lIwNpJJq/SW6muD5Qq3RSD7Z2lMb20YmwonRuzLVO2TWWvel/G1Kb2PQhxwTjvk95+FiBy00XNtg1jMYyIYWQTNYlI2Nt3SnY7OOsrvYKX/dw6XIfCI6Gak+4XeknHx2j68a2mCsqXlk/0DeXo8039sPmcdc9EDp6qmDi4qQjdeun3wSNBaL4oMHQFPIL8b1CaLyi6E3bW05GXoTb2hRQ1Mf8+vP9CJaJxfqJb8n8sBZ/qMlQVOj1aJ6WdAhSTnUf5QrjOVsSkHpvdwwOIHePV2Kd7aBKt0JsBXkk62VfPz8wq3ThoIDzfjH6ZFn1p/3GS67Bthnpz6dJbxyEsfmWwkDc8t8iH5+AFwmQow92T+gyx/p5cm3x4xbEHiuX4rvoZGm2ONTskqtc8QJ9U0/OemmbxfuXbBPoYXes2d8Koatg39Hptc0Vc+0nbp8fOb3cXxSd+uT+ORaBf9/+KTs90fwqXdUpLa1T45SaJGxQMqZ9zgoh9sMfXi4FCBoxLn5bXYHQhJEYVkMYR98fdxcTWhg7xP8OHzm8obxUfZA4eHXsZ1BBkmmWpUv57+LZECqyTagCuqs3QV929SVTfsLBq6LOFhlcterOfEVPOglbUH3oWRCaW4oqwEgaWua0TX1/R1u63/upwsyGiySe8H5UR5vHYEDaEy+D6Cxdv4AGt7JLsf0IBIQq6eSaK2YTo01C7U8AQ3nn4fRcAJ6GNn9gGbioKMRXTRP1tnkmKtmnaj4vmiMOolu+9ga1dOocpTbbo+SURk9xYBwujEZlxxbJt2YjEsGQ5uBPSsucyi4h/G8oiwm9GEmDunB2AY/Pwie5lXj3Yha6O8Lrnw5zV73DAuUhvhtpwr2DhtmrGP06jmYz+PR0HUxSE3HVud6GKaiA/96ZW9/se7rlb2N72nK3hr671L4FlKldesNsVeqSy74+1y3wbsIOuIcP4Dz8BTjsb0JqDE6pEMKzuKxBT6oa2VSTytWw16gwRNrauvDN7E9zPx1RAX65sgQP4v2ytgxmzZLuRLQhXzzwCz/OafAgojd2wCUqzn79wuO0gAN8dDFYRYJqbUMoBxapx5pzrEXiZXf23fJIPEdQ0y8ClZfImYeSuWqsNwozwNiV8SfDuzKJ+kdSouHDWDcJPE1KDpe5fHAa/Xxmq5pNIXj7nOoaG1IAGMMTQxd9NdMUl6i71fmK+y8c33O4Y2hjwg8auM0mkMb+JQN0ggftAYe2h7aInPn05Lq6xwe4Pv6jcDd993P5tKXgGWt8PPum76U+pcJKCx+5/iG76yBr7CYrzDAjsFk80p0PWFtSSKg1K8uSJ5VsbwHmbNOeT2vc1Plsfy0yR6QjcA3umO4pUUvbUsAH51P8ZY3BX891Pgq7597H+X1+4g+vJ7wp3q7D/Z2C4icl7uG962qy2tRJGyKO5tn1FhAp/ErHILrXt7nL6ti0t34LutQRtfdgn4hwgFcfeB96FSOfKAMwD1WXSHFgZnNcothrsqLrc9SXUjQ1V/Kj5Q5lOC9ita0cc0OT6Xkk1h73cT8+eqYPl+N6t0tTbWu0/GA7n4sIuuiHUK0yYptE3eZPocVtq/kKMj92svWcr3xe260cGBudS/1VkYde3b2otrGGHNUoY/+uhxzJ5YpHabNSYwMCZJrHCnR68ekhka7xlZvtI7U7NEHhhLlb93qn9a9jHLAoRYUB0AYba6wQoF/8FtooSfAxG3j8sooBjFH3oAJKmIsLTO9Pr7QQ8r9Fu1iNsa6t//GgksqwseXRfxtu568UiVWBo2x/FQEy6nZIQNtYIxbFGTtJew7XD63gR+lzbVvfg/aRSw/Do1Vx4W4yUFsF/4QWUNfjqa1oicAk0Q/ZGn7GzUEDAeObSbyFdVYLDDLbvu+06URcmZMzDz8MpYVVzQ76/LG9qGVP7CpuLj/7uuMoidcwg9cwHuCOHwSdKbPX54pBQpPBLMc0IS9IIxloY8eDtqYrkE4+gqF9dXK6uGoTWdUD4x6WmmeDfO0sjuzaswQZXnUDFm2qyI/0gbP8TthWibvncrJD0aZsse2FRZL9oTz3vy+GZY/JFmOJW5cosxfygCTSlHuBk1+3YFxunl7m7mVex/f/v3tm09vf9DvvMq9lv9fDIb1Rqaruk0MW84shRqNMvwWJAp6HFPwPY7xLYs4lgF4fuVi9L9vfn1CRmMAAA=='),
    'finetune_fastvit_wham_downstream.py': ('a5746c88555b9048233af272c4ecd135bde46188ca02ebeb9bda915281b86c04', 'H4sIADdgoWoC/+V9bZPjttHgd/0KhqmrotYUd2bWu4+tWKlnk7UT19mOb71J6mpOx3AkaIZZSlRIambH4/nv1914a4Agpdl1vuS2XB6RBBpAo9HdaHQ3fvub54e2eX5V7p6L3W20v+9u6t2LSRzHb+q7Xds1otjOiruiEdE3Rdv9rXwXbcqdmHWHXbm7jrqbpj5c30RFtGnqn8Uu+vufX38frepGZJPJu5uyjeC/tajKK9EUnajuo1bsC/yJFbZQX0Qv3vz496gTbReJ26I6FF3dZNG3XdQ1Rblro3oHterdBIvWm025KosqAhitWKuqWA470+6rskuhhUqsujZa3YjV+31d7joEQk0B/HJddCWAU4WL3TqqRHErWipA3aBP0WHX1QeAsY42dUMfYeDQ9Ovnf4ARrcoWoNAg+90yPdqUlYhaGBCA3x22oilX+HH1PirXbRa9Le5oCJN//KMV/zqI3Up8AzXa5wTgH/+A3u3qjvrbRjgF0ItGQHdEdKB26qgRq/pWyP5ti251A81Oym1xDe0qkNHVPczIFvqGXVoV0I0i2tetgB78UMsh84bErkOAagwZ0sJkQrOV55tDd2hEnkfldl83Tr3JRL9rrgkP+nnV3uqf/2wB8+o39PZG/96Xq/eVqdDApNRb/dQervZNvRJtq9905VbIDsFcFquqaGEsukfmlSyxh1aA+vTXH7FRSXn3e0SHev96d2/6r6hQ5Hc3xTbfiIKGDN1ou7I74Fijoo3oY3FlRlZfQTP6CeZ6f4+ldnvT6bpZ3TgP2W6XbQ67FUIE2oHS36iu4Vfds92Ovcyg+arNcIz6+xv4/V1drEWT0u9WdJPJ26//11+/ffv1m/zd29ff/pD/z6//90/RInqYRPAvvrqqP8Sp+i2ghn4AKtU/b8u1/omUon+/31+Y9/+klfXijX7eNEBZua0HhJYTlZkCEpPm+VrsoNfw9Gj7+7fX34309mkdBPLt8vf7F+4LXtrvMuvh42Qy+W9DS4nkbot3zUFMJ/QqeocL5B0u5jnVpnUNwOYR4IXe6AU4BxbQ0Jtyty6BkudAF9luXTRNcU/v5aLMRdPUzTzaVHXRHW0fmMdPugEGpA+c5ngedYd9JS7ttzTKsmxJJeRMmDLQf/VxMlmLDYxDrHMBXAaEA6yZBJ9plNNo9ntgITvVAbluM/xMZab0FloMf5AUvS12h6LKvW/lRn1eHdZFVrZ5cVuUVXFViWQqG7MQqAgDkxdVpUDJ/hPzW3U5klKCixtnAFY8dd8iRMLFQkB9anlnXZ3TYlb1phkIwfu9SKAaTdOLC9nhRgDl7Kj2JaAujS7P0ug8jWbnS43Grngv8qa+a3kf0hBNUMfg41zjAoQNyOICplrVTdXY34ldWzcMJaofstSlLIT8Q41CNcZGAdP46vPpdMlHAa+Llnqih32pKi4NSnddeX2oD22u6F5+T2BdihpWAcNwVbYdo7ul7KwEHMS0BjHNYC3eFNDP2bkhC5A3ILB3umO9kV/KkVzVB2yvpCZosopuV+9+Fk2tql6ez5fRbxYaVXOYqWn0WXQuVwSoNXtVldQBRBXoLbtrkbDW02iNeFwYPKas4aliAmIHDEB0uKyWABF+JvIT6hXUENCAapHRtmInUAMq6j5Tqcuz5XJqCuJS0WUBDrVnPsplWbYiegv6DMjNr5HFJJv4nVZRtDoSPWgoj6izIZrtJMe2OYSfFet1oos79C9HoYikqlcoRjUbzEmlgBVQdwn+b07SmEgEf2gmBoQEfI/mLbGTC+Wj51HsqEkxviGgioGPlzy9LupkT6zC39vZNaPBmbFDm/PJM6+Rza3LJpmSUroTH7rEfruu6qskfpbt31cxUBky3ameJ8uBvbVgqk8sHeBQfqi7b5BMJTGYmvEf60O1JpCg6oJeDCt6Z9TsgIbqqI1XoqrvothA28QPiJzHWOFE00SxthQB9FFUNUoU9cKjC+Ap3SUIz5RLu6WmFKo7HyiEGsSjmQhUA2l1gMpEgoa15+KWcZS7Eiphzazew5qPm6t4imqa3BW5GN8X9zg0aFXqshk+JbJkCir1ql7DalvEwIbK3TlbT0pqS1ajua6CdsmUqCXjNY7kMWREcIDLlltkay8ioj/5krjo5exCcrzk8zT6fHoCk/grEOEe9lJAAqYjEcFCXD4gah7n0YPTyiMbG2keMDJSKxKnvZ6ICQzvcn5+tnRq4VQqwQEdMFiSWqwtansgNZscsI8SliSRYsOXtriB6gBVSlG8dBEVkMbA9O87RxbpfxKqkjEZ7BqhH0lctKuyZGgifNTNFjanPwskISAbCXkKZHQnmsQtyweVFXugzXXyEG/jeQQ6RwxghPq5gb/n+Feol+ePl7ahpY8mO1W8BYfKUPYRtqdISPikqp5CTj9COdg8baE+7WC2ZUtbVktMzrqg5X1J66/txBYnjS3wRFFdKqks1WNwFAXNIyaj/YJNsNkrW45Gm/cWiAMYJfXQYRq6q4bTUkOKyW2Lfa73z1I/aiX1gwYrqpazLFCSlmkU4H8TqzvZnYaiRdiRv1X7fm1VsJYHaVxAPtUUd5FnCtgB0tqMdvQMyYDZE/iyHLDRzkKqmxzfJe3Oln31TXYytyxvEADje4Mqt4KmN3Gj8MxOzwEndTYOTHOsQUCK1RzrlF1Qg5DU9neoQ5Js5j0KsKzLUR+PK+TTYd1SFwmpl/umvhI4F6UU8ySnDrsSCKPH0Svgi7DTEgkwH+INCA7oGJkQDH6xLXfJ+Sv+ra9BB5i47ALunKB12UWvV4yXM2rwCeTSArIVjIRbeOQ5WpoIxVYhqjDYSyNHcJk6csYV0l1CCWGerAGaAuTOnEgtRfmwDMgwvVpTufINV8pKYJ9t4rFomHQgme4+Jx5M+naxu096Mkz3euEPpFeS1NY9bMU/0G7pqiVRAdPvYG06jb5aROdi9rIHAAeBRTQzxzH8XO4TGE6muDz+dDh9n1wY/++NsW4I+YYkqLPQo99LYYbQ1fQHpDmtsd1BOB/qq1Y0t0Q/rPalacFVYGhOoShNZOIiy0CyCJtOp/3qRuYn9Gilx9QR1Th+RULHRTOIQKQZZRy22xYyfbtbRbtPjH1azVC9Trj+13a510kYvCx8yZYIdNeWjX5P1OHp1424LcUd1I5BocnQ+AjdfkB59rh4kIaz7MX1Y0xdVm3iV6Qh1eD8xXJ6DBV9mmS7IpDrw9ggM7s1rv8OVBrZ58d4gFABI0VV3efiQ7HqcDlTfwf7j1hCOpFImjGMwYpClH2x9FU1p4UpInb+EQh4vb2SokWNb4bjQ2SQBX2UROZR3AP4GZtCt4MDeJKiUFO9lYXGBJEy1keMNOXIcRQ1CcsYNpV6g0qmUtFIYWJ6WOqacVMrYRxDq9XWsJLS0wx82NYj2zego+fmG1dXeXljDfCV1/4GfhN/X7Ytor+v8c2Z3grg9JoFpOdUUMsX6nX0C9kSoKv4R+kzgBmnjKN/QINGviKlWuT0JLKSe7yGnWLomqA14KDsORCfRNqDrjI/e7l+zP65v445rcvqiDJU2n1JR2PQ5COLuqTWY+y40gyG0MzSN7Gw74ueWQHnT1oVZPefUY9d/nyGYtAg5ytarxbo2BBsKSNmlkcGdJx0CFJk8PzobnkY6aiFRD2CdaQOQwSeRhXVH6tyr06hEvX30m52uM0aNB05RFyIeU5nM3nCzIzVxlrZBvZNHofoK8u2RG9ZWysPKuiN2Nf+hxWMJYdZue5u6LTDfoEegGbhvQRJnmOV1i+sT0uUcdA9M9FjzeQQcSdGP9yPcny4PZfsy/nI2ArnMW4hM0q9JcHfbhE2XijEntxiuTqNlSRIZn7OMDTmXMWVjpTQ8hJQW81+RXxI9QYWZB7tbEEHkVzeXxCA7XJ72OZtVzQ4blw9VDBTGx/cdiR8SJ9F59OeMUlWh+akaR/2LS7gz3DrIqc7pBDiOI025oyDqrMFr47A3tIfeTqVtTeHzQbYFYFxeIOhJRDZZ2678rWcH1BpTMmlr/vJSfgIcY8+AVqey9bqTfTgofJRMow2uhONiMzp3JCqYwjMdH3CFj+AhbWPZWh1AJ30jnVwfi2Qqa3tUKOFYU7PFEYCpBvm66GCdhdPZpJ+k3Z5Tf2OB+DxoV+LDndnavh0Gig+2MPVAd7Z28ozoiMhqhF1SV+WbnFdQi6sSwZiyc417gy/0UvqUsKfR3pt9NgGEAZrq2hgdHrfjn/wyC24U+8ZSZwpsQenjFNym07K22IWhLBlRR4PfhAtHhQjOfLj4YG2yAsB2qEGPDAAwoBzRkgikus0Us30mLbieZlVYQ1mpq74MiyVk4LLT326pbZcep06zHdPKi6eeau9NvUxlSPyWB6dh+ip+haByYORPR3X4MFIfWhWos8nm+sr93xDFsxATbkVsGOM3/7pDzE3BR1gur4YMpCnUY6WF2dQoSZpEJfzC5g2ud3Gx4vlFNTKi7OzDHj9xctX9L8xQwIhXvP4XjO9A3Y2SN7j3tmGv9cxj+/FvfJZCyyLAeokr5c+dTairSvpsHQ6rEaeNdFynZ8vp1O+T/+ANC7xr+HZUUp/qXK3P3QuosyQUrtQUta7gMnvt9H3dDKgdMgXbzJc3gQb3VS0eFqLTqykXfIAku0WNgDdTbM4y15lTC9r36uVbjoiXTQulqB2Q1m7ykAJLYsq/6c/AbhO8xU5xuWrelW74zsVtcZjahC/015XODkA+cCKwTP1HepFTh8St+/c9J7KaYN24CdoN+3iLMgi+6eKCiTWyUE/qpCj9Cj+o1BBXliDaMB/Yx2j73qEF5+n0YvpCArJWmHncluAOvchhx426qQnf7UOj8IOHIvLikkfK6HGO1GsboBatEvZr4Y546PmSb3pydib+hL63zi/H9dDZ359JQPW7dCMq9Z+5QnvYSnUtNL2HhywcvffxnOFVdCbVu8TtcFP3aIfTCmG+w9ap/BLI1MLVdDMLlRH+UOGqvW4jT27g/3PF18EQdEcz51lxmvhLL3yK/rLAjvjvfJr2DnFwvbJlnu0Tmmb8pqcdTtUCrb1WlRzM7nKf/3P37+9+Kk74MkAaczX0rDn7culboSKWCfPIQhYZl45xynmLYz/X4cSBpFfN7BRSL4pqlZMxwC2oDwh9m5EsX4qdOkNqjc4HWl8iyheG8f9vIIhS2TEc0/t6/XkCmjzCsafUfn28sVyoD8n9umUpsijPkc18OPb0qbczh65YyvtJ02/7Cd6git9OTBb1FzyERMwiG8OMVCQYUuXlIMHfQD23rl0EmZqGOGphJG2yU50d3Xzfh7tdtn39foAXMwdcRzHryv04Fod3vzwQ/TdT+++jwhIZIDQBqA+dFb/omgPO3OZ8iogCOtaSPc0YI3oK4YRCjB3eMQVXR02G3ReEGKt4iuKXfQWKuFY74pmDUBbaea6uxGyLn4uye9HOeijXosYyqK/YIgIRUDU+HYmHW0aOYgKfUCVkollWjn5M8VvzPCkp1f0I47vpug0BuxgsS3oTlkJvaBXFEHQYadg+OjETz9mVXEPra+bek/YqiN0O02xLqi0+ijSjkJ5ad8UtyVhYg2KqthjCdHcezhWcTaZnjOJcTW7nGAVDvJmh9q/LiARlKuPsKCuYUpgG5lBMb9apgewACX5rPfVJcFV3ZY7YLwYMAPElbSd2EurJ4y/Q89seKFMoxHM8PawZ6+IFkkNmNvlJPagnjtFfWmLh6fUEhn3YF+HL3iNNEL191zMlLoAm0YaMPqWUr2Z04AGwXrslUi1EVH1ABDzEho/y758GT2DP/j/5Dw7g3cY3wL6OvAh/LEv4Qv6P+guAKTsDM93Jf7U2lXzg/Z79P/kk8uWLuyjkIjmQ+YheRrFX80nzDpAh9WaIvaNUK8TAnoJmsgS9C1gDMlUNXQp9Q11yJCnkepfn7DI0VE0iWnJQNDax9L1m5KAFBIkeZF0Hxz6RJ9vQLW5M0b5SasRoW9cVfG/D6AMF/Q1mqE5zvTLuklUT1LTriI0VFAQU7ncLYfXX2Khp07n3JMWrlg9oyfpz0mmDalmKQQaBXdTYtBdd59XdasONmGikVyk+hvCjtKtwh+R+ckD0CFM/TZ6qz3nrkUNzAp4V2nC3vCUFw1b1xgRhweFuNjRF6itidsBz4XC7RbKKWjY8wjqrQ6VhKprvbhAzrizkgGIrqpbdRZN0uOAjqqvv/8x80bubOnNXoDp+g6S5MYkURsXiZ6jEAwW3dqNwFHcCrJcme78t4KKzHTXEt3PMGhE7YUkU3Ud4RWcbF0W1xg3lqzL7flidpECM9heLGC/n7WHLe770e9K8e0pFL9NLoDhwMoutnv4jPYv5ECa35J0Q+1FNhr3mC1ytZnqUq8SzDioXOWKVftt9Msvt0Ikb6HW2//7bvrLL2h4g8H8AhASmLeumP6SRUVX7C6i4rZGSzC8ZfV3omhmKDqBMTTlrcTfqq6qYt9i0GV0PkP+KkHJUE7ZVxnL2YLEZNBg6waU1ET/OhTkEyRDKK/uvcDQ6Bo99FN5JgRI3rXWlNS+Jw8UvplzjTB9M6SaLmVwApQvAR3uW9wrLdMjVc/IXDXrAzw7WvUcC/Wq4vR7Vb3NGtATkFMa2OwqqpTiTmKjQt30OrsVK3jO0SiYILZSBaW3UZa1aPITBJcqypqyUIW/oQu09UVvD/s9nbMbNkc8Yh49IBni8bSSIkz9rjE0NLEa/Oh2YFTkjEtcWUaplvmYDLoT5fVN55xlE6NQIAxTwa7nPa5Lh6kO0KEeqTMistmrcNOa9gN8e2XleFj/UMhQQrxv3VKWlp7cNkZ7Vb/nnIAi03mptISefWAZLGbtD0ufPvUcjHXNDEt1JVsDB1ndoL7jT2EaaJIF+TB58bNUKt3qsOzkJk6Jha0odqRl8pdtt+YCmAAN4uNpEFV1h6roMG5YT/CRmDqznQYolOEDva92IA6bQ3fzia1qDDD7z/JI63K5m9hhaT+q34tdPI++ybatkA3ySTMmyp8Z94vRSUnLQY+ra5Gq/80Ast78lNsSJAwMrC8IBukqQPGKZw6cSk0znPKEd1eeTKDr8mCfB/oZRoXuQRrBlmcB26dXo80TS9UGuhDFDVTBU5WhKkpILQeqXhuL4DDFDVdVDQ9XDbb+qKKWYW+IJ6ugZCleTn6bSxCFMtwHzRn4JrVBRZIwtR+265CI8FJVQseP1LciJ/pITpE9wORuy5XZ0cinyXH/gZ7N+r24n6tYpa5OJJw02gFSrqp69R4D2Pr2PajERkpd1QOdcOustrPkQHObChH3q8jltTwhnIdc0FSJAHZ+BVns1zFmtK8/iGaF+gu6UoIaUkV//Oub165py5jSDntUO2HrIkpM7qHYuA3QCZm0U8fEiLbIWMcah42ggxVONxxOLSEC8fsEiv8eGPUcdu2/DkL8LJKzaYBI1JxdwgJTlPKoiZhxc4kK1P+l3kKjq0G33AlGhkzBKQ5dvQLysX2SMHM6fJe/M/xtWYI8l1fnIWpLav2zCTtrXlPukA7rQof6en6u5AyiFAZ59rNUm3XmGMPOBGmMiV/DHt6m0QvrpjC1pzIue1cuI2hTOrv4PKCwIwFLQ8SAcuxYnNPI1fuscSPVS2ZQELvCONNEn2j1QJuSF9Fl4IADv+t9cy967MRjGe71NXB0QFtFt1UeS00Qltwx2/R7NJIw/pMxJWsWhya39WElMH7AJFAy0GKLFdoymb1lYDelcl0AZkz1qRPrKAuU7QY1VZEg+qcY7xL4ZhqkArb5rxbc2/CYt+AmNN47THpT72aypTlRweIB//+Ykg/P4sG0xwIjzEgwBGl4Zvrzb8yYjAKm47P0DctUdTARztU95lISJQbg2Bb8qQIKOlS4X3qIcUyxStZC2Da7iCmG3MJc35aWkdKATXE7AY9PYHW9Q6mA2V/JdNlPnXiE8i6Ifa3Z9SeK3IrSDc1Z6qFBH29Zod4D+sufsY78TC+yv+jXaveszg68UlVjThWa7Ltie7UuvnuraxSseLHdZ0iRP9HbY7LfHv19mi4gEXIt8+MoP+8R/WBYRFOHPk4skw7Zcqok97Znzyg+ip6MVoqLRo31URJfC4irSBqdacw00t6Niybbi2YDG4sDbjFY4hI8DqFwPKkHuC7blUpIRaAW52xBBjQIA0PrAFO3NOxXftZekcPydX5hBayht6NL6qgGcZIWcVSTOFGb8DQKQnVVdDbO5FRFYTK29TRqB4NtpW0f5VKrcGFKhQI9ZwGfUp4f0SysdtF7HbQSmelPj++leyW0ltK3jPaWcTrksU5cJKM/Upj6ygwrBfou/sgTQ3nTvnbjKj3Op1P0mhF9xgBdeqmxgHHLjHHkP2WkTm5FbMtOAfiwYXWHBqPHi7sW4eBBc2iqyT4o5vLZgi1m1ktkXJeSay2xUEicws46UDewzbb0GA55Vs3J7bppTCfnGG1Nn0j/D8vscfWeRYoV4gPGKEjO53uwNBj51iNGzIaYrQ/bfWCp0I4q+JZ2mcqlRAqNdKyc2FMxYNbjpVBqsAGMlFYixpu753qqR2puy92hI6evJCBaopkWPWjPfIUiLAjqsfc20NtNdWhviNGHFri2fEjxKOnHDCBEWXKwZreKyfmUTr3biIac+3HFAuGjwqVOk0RueeKvo3iNhMGFA9z6sVrH9CIMKNLhdCZoDV9KYcBe6syYdrWwj90NCpS6Wgc1Kk9BwlHMT1ZxdZpTh51S+IrDG1RMDGoXOsGFp6Q4yUr4Sk/cSN2uSTgUkJUyJVycx+gPgo7VJlq3v3fU3R3flPxQ8zNIGaZDyVqUv9CaYnspEZeS/vbk1swYRoupjLa7RLeLYd9sUq1cclN6mNwdznJRiTwsLJvLw4D0U3gE7A98yvTvS8qCoToxXUoRqB4pEZNN7mFCln1j+XhADNboH8kdqyGN4KcUZMcip3fkeGngihKzqpi7TrwAbKJyUhh6ZGaoXiaUcshdxYVNTws7ceNMenVVykwnjsoNnOpVeWLSnl798dAWJ5ZF1cVVaqqbeEXUhctd4oeWKmanV5d6wtUFcAUtCesqTWvC1Y5P+I6IGfhk47/C3/uhABo/4+XlGVq4rMP/LHpci1AwtFz2tsVkgUhp+Bfdvy0MJ/BWx8TpOTOxcZLlskQM/cjt1JA0i5G7ZO2HAmn6B288ekl8wJXW9dhEyCraD8j3zqYxvoz3xqvBRbH7qS9J08AGNP23hJkME5MzFh2u0d+8mfiNofNKP3qMBWR9ZBAZIPpI/NivFcmlsWNzSVv0/EfHbTkDdxnHf0akFg9VMdEqXiTRx6PNw9ivtJJMnw1z7QVDgRDeklD0IpGUHg1br6C1ZSz8iHd+6hzt2VPiJ8cnOUBdcuEtuF8Gm3tSaFOYOCUZfPHFk5oMhUCN11CuB08g8RPj0zDf+NlyLCeaN9RXp44UczFCm7eiMvhFs2rrSHmAp823ix6Yx568qg/d/uA40x6UoRvvDklC0rFd+MJukBajZ8/kOhjRBz6qB76a8FE98JznXJRo1pExuKv9IZkGvaRItefjOV5dN6cz0enhYz35Mi+u8eIVHvHo5WNw/MEk+Y1g+hNa8vzQhlpaN+Wm81ugl8mwJxvTSfWGKxjm37fCmbTgsB6MRtovJXcNUMYukkAhpfoZzy1pkwwavk5yNBvWe3uBliE/M9/N6xRbW8zRqmZ2La7NYBySU3BDYPhcBcA49DQCRs4+xlVIa4oDZDKGKw4Ztkehfp+ED0t/Ttv0JtzxxwBVhwwemkL9926tHptStfz3XlvS8GGaoMdwd0jQeV1x15Rv8vC7YEsrM5m1ehyzk6F7gdjuu/uexUxlI9IX+KCX02R0l4TuFCE0TycDB2dutTA+Pd3RtCHRO52EZECwNxZHHs8P98IW/0QeP8jfp5nUqdxOfWQrg7zdbeXjeLsDwuWxAGuUkT6RcbpN6aK4yE2a1yD/c1DoFA/yOWW1N0yNSX6qPHOa9kOOeEUV4shKU4TkFwzdbn8ctiUdltT9HMzNurii3bj0tOfcF7gfezIVvlqwIwKML/yQB/n2kntlqwGF2tDfxhoYrs9b0Wh1hBrhAIdi0DPWkK5qRcCSuxD3/G7jYrUSe0yQDKO2TA1Vh6pKCNuZvPbAERwxfUFHZgpZYj7vxAzVgZ5hrLym0U0ob57k9Pyzs2JovEaKud/64syvvD97aerCFhe6g+YXTNPpFkyjl0EwzbYd0ItUHNa/mi5h3v3H1y/LRMIbHNRinLUyOa6u8IXJyg/oJezNE2n9JKLVrRjCPa3W/suXju7izhyVSaMv3RkzlAa1zO/UcQK3xJ23tM/CONFyFbgSYewA0Q8XbxCie2GQgnsZxro1YD1/MhsykEfYyQj4k5iQaePIvI41NMiEnCNwObsoESQSp/6laXbGhuZq8DD3CXzODHigxDLA9Uwd+WIZYoCmjHqzDPFAU0i9WR7lhLZG4PPyFJ5iOxYusTyF0fTIZATIAPc5sk5O5EinrImncCsNbxLcHw5Us15QgzxHUhfe3IPxFB3d36O+BakankxCClt0mEp7QdNxcVeUGN+Rm+tk8xfr/V2Ot67GfJUA6v4pL0lwYerlaC6zzdUNRZLXqZSLQZeQE71NyDGYuW74nrHDOFLNqPt8+l8HOYO9LQu/qnSQU/5Ju5hNwmYYQws0jTlCQbKRSXrMq8Q3ZdJQoSD9TSenulSZ+cY4QkzpFPxsySqen0h0fv/2N0UrXris0T6EC2v0y2VDP33Lq5e4mxCs6OoONj8ivynxkuJ7yuSp095jzrawB4KXOUht4Km8R/82Qoddo3YXY4DJXQUMcxHH/HK0Q7eZfRG+Y436ibvMVXubvYH+/J1emBvWNqWo1uQws8AuJ5RY8GzJVD0JIaM/GALlXKvFP1I2O5lJVuIIPWNgaar0kwloy4AYfctx9gM2ip4zg9Qu/e9Ngm95VbC8IA5hyUw6ubwQZqqX3Eh5/MpLb2UKd+lXbrPRhy4BntH1k7xDUxfGrQxu9CCYa3llfds9G4LidqJuOMSnxJTofPTqQqv34l4n/Fw8OG08pmxp2G/wrh9XwjOk21TOdDul/aYmA9Qo4B7Aou2dV841WHRZ1uDFltYAhsDcipjcqd8EW6EmzXvoAjE+ad6NYVMeCQnVQ9n4eQpoC8fLoB/Kl5+6QzKZrr33PEOjhwRKn+69NHnL/cJCrLmaKn0yUVCYmMGJc6kSHWl5xgiTdjHpdc1xk2cqyYeB0v/FC6lTzUC5c6ecPY7spU90zg1DaRJDKRJDDaJLPq/lpEkMVODtSKkA2j1dmWwQSAGc8gJA6RNNn0IBnHJWrDus4iD2Jikvpvgh1vOFAhahpig1sX00cvCOYKJfdJ6aPvajjM3Vk97sO/HG2ocpCBRvLaRXDA+Seam+H7kqkN2IafLCKRKVLWE2EgNMX5khS+QD+cBH15zK0X62tPnBJ4PeBwNZ0/mK99KmG+g6xbtzR9KUr0Rk+naCMfx9Yp1YfH/Z031l9QRzH10rX5S74sSJEXR6cixk0fOsVVKVO9WGrkh07zrxWYs+xceR2hd8OQZEgyrvMm53zyr5Pc6Zssz10+06Uyk9l4NAHBOgukEjVI54sCqmWCwvx9RYCUQhfk1eD1ITUFPh2O7Kpu2MLWiMhmO1NgzLRuVJZZR3Lu5lS2jZry4Xn89LHNb9QV3sl6uLq+Q+GbVmNIfT8a8qIA80zNUkKgmkxeCjyTlabWgjl/gasbp2zhzViHuRvPBdjmTM1ouB7GJhzxAN2cbhUQou6GGiE3elLBsXi0MZz8Eim9fHMClBZYd5bSuajsfmqGvz6Nqx/+IH6bBermTsoExsphIJ9ouo3FQUh3OWnfcLjGeWHRiHqe0NRZmpQyFcanC23WA4uLkQfvCiTTnTHXlzJ+SNo2eZEnwpTVm1dsmuxwxcA79EvnZ5QQSiEgfIGKL4J6C5GdIcpZHAm4UkjVI9lEbo/r8uNxSV0pXkgCsziOLRIcXq6G0Nadg5agqSfvs7mrm90hS3XqbA6+b6sAXwP9KXhPNxIFtyrcRboRa9Cm/EpjhUXftnUe2/0YV5OiUCiNfLY7+oShLPZuivNgORAbRNkQZqgyrpY83COQfqk8I9A4V7Roz4Y6BYTjKbaS6moFlZbaCaVzcw0IUrl2MMe8bMfcA/MAyvE/ISeSQspVGYi9cDd67/zrtRL14rnGKe2d5Qw6J8CE0IfyY3Xh+LahAInwhBGoNm1uD1sYCQk85wp/JJAD69HyhiZ1IN1yAoC66auIVzdXAAH7R9CtY8fzVaE5j0TMr3UOWz0brkij5DV/Rg5Yuj6xWvTg/V/Hy0JhpmZmSdaz+iXcy2PSP73RiME3pQGVajbtzVdV+K2cvR6hsQwd1hJ47AuTgZjs7+PQprnA4oBdpMxkQPwDjLLsb7A4ryTCoYnwhI5yr7daDREkXRNg7n4mV2dgKvAI45Duf87BQ4190JXToV0vFOjULickvrTDNSA9hFhzc1ajCLsArJkiWpBk0xV8zFOv/vjJQHqXFSklXUcGQyixnpIu0BtK3bskUD9wlyCWWK2s88nZFhZXWuF2S+Z+PVlRXmGD989flRTkwkwU7E8PcQmR2hDQSnT9p6cAep5Pw4UGV7mtGh2kgHjy2nqr6eUYx8EFkvxyujCTDM/M8uXp19efbliVpaV+9nmCN6Zk5vmK2Rch4uYjz2wE36oU/NP0H9qNjgqQNmCKYtLbckWMVA3hfQwut2Uwqdtx9PpU+hbnW6MKMznDTUsyPYUhuCsbrm6IdAcMVfZxykXP7uLhYL6CvudWltXJOmWrX9dW8yVTvi0EmQ1pnoll7XX6N3EOLZhe2Zh28wJnUxt7MRslePf6VrUZ9HcVVePacjxPY5vs/29878qRMGjL8N2yi84WnHVjz5MjcTL73zkuO3O9vLnRX4iMDPo9i5SVvBU9tMFciw3ZZ0H+ThSkUfZoQJFSxgsX8ZX5dIQHEjbqX6jg9//vo13VW2ulsvPNt/hGlzSfVVTpCope4tgfD2f2ONGsj98z/+5fvvv333lDOhr7XNlaSHAvsQAPqYwoQA5tRHWbJ3HGSzlS1CZ3tTtt9mKToeYlMP/Zv070d5Z+cOddOUpZ1w14puhs7JQ0vDSeKGKYGQZEzgQHIkpdlrmWLyTz/+FQ0XllCAOG0Kigh1yhkolUAm2jMaea3MZtJh8HhizmCmLE2EMWPJx0SmLFKEBqq+ut5ckoh6zig881abfszr7Xs0/QKJYZYWmZ8jEh/Ktsvr9wxv/+4j0/8/TgRlgf/Ic0GZpiavyZbSc0lzjwNtWHU8l9BCgdaxuo4ZyrhpY+LdYZvrLe3ckjkmT2L+VYC8rdjWoPP0AIC22wKJo79KGA6mFkjdIyn/M7fk8qFfElsSOJ5NgekiYxz9BUMSPNpMfYlDExin5oKbMq8iJjZtZkrjeTQigYlaaUmQOrLaH2KTOTSvd9X9grxo/FyvqW7XHK15Fy8rp59kqFmePk6FufogAhF+LvlaxpWGFYh+otqTklI+7ba4QNqbJ1xJPTSiqZeax10mg86vc7s0Q9+9U5YxP0EDZ7CQB6zvxMqA9D7yJaTILZwMXjJdfMhluaHc71TQvumX7qdel1zPfd2vx3Om2+kibITLqiNENrWYySBc1mZHt8XVu+EavQbUO6fGo+Px5yIXc8Bti/wWmR3N9jmfTNHd1OteZvqY57jUZgFAG3W12K21FwToDm1XVpXyiXTyg2wGR0kZW7SMDYwJv7s5GjG7i9OnTonMIx1xzn59xoSnkRcvX7Gwb/lCbgnCzCwAUHNFPEh23/izKfVe1h5TkXtln9JNr0bvRFnpWCdAchQ498D5dCBMq3McaFqpaOdofOq7bMZMJTHrNaSlxMr0Pg/rK5922O7IYV6aMlMq7oWD91NZxk4OS8Oj+EvetmH2eBBvHhxXBUKSp2U9Ttytkkk9C2t+KJN/KIG4p27oOwVMGvF+553bVXp7sH7bfDNmejm8K+OHqV+rKxhB00ATz2G3uil215gAGl1rZzAEbuWpdzL/mrUBZVkW68T4rahkcOFg8r9AJqFeLg67ixlXuQeSCfmJgswq8dV480GleHU/jOUj8mmIz9QkkNnyIda4cd2aA/EluuDUmb3JUEpHhXlc7cwstzBzYT9TqJGdGht9pIvycCMGlgw7C7vHfW7zet9sm4scC2V74CJcaaaTJ7IsaMd2xjHJEU25oafR2ZQ1putxMGwI8nZZ5yIH3ft4wqLhilum2g5FDRh73bDxbCDvFe+d548e6pl/e5NCt9tKz2udTbqZBz7nyl39WH48ebGuY2ZMehdhKBaEv+WoOOtOBu7yVZXsG7eqWhPIRMy06tza9Z5vb+w+SObUxtUsoxPopls5gjn3a1QfT0jMNnB3CMszLo3duH04IUexa5N74m3WvsZ2eiLjoZuVafJcDBgfncuHmBpAmeeNLwUB22hxR9+q5tG2hgrg3KNZef46hqNhPJ1yCfYgnk7ElYsvFxE9eCdiRh8/5xpF/SwX4ezEFnwAceF8vqFmTeWq6VfyOhPI+27MFfIKgdfrYvt36W5lrjDBO1iLezw8ZenK+MW/C3IMZCmRo2dq8aU8oSG/HVhlk+RplHv3CT9/Hp2jZ9g0kDzb63bw5gOXgdfuFQrGRESFVY5+an3B+qAvYF649zD7VzgH8q4bcBrE0P2RfsZwMzL3kgZlSk5Ncvz+9VK5vIkT1LJE8Vgvtzbjsp8tonMvf9Cxqwwc66fkyb3rMj4lhb26CaH3fmDiHHIIfSqC7329j40/lB58MD3+2FbCKWOSoKcjCcccrWxULf5Y7IbU5TG1+YRcnGPoHFSnj6rVJ6vXQ2p2GMODii0LGuypttaF9q6XJ9CJknQU2s/4SgsmbxrNSP/s2cNG2QgwzuQxnrPb8tzgEr0U7d1g/dZumUXGDc3mYbpHorMdaIOR2gxgWHwN1OwVHmp5MLz7aMsDNU9uecDae7TdYL2TWx0zEB9terjyE0fNDMon4/lY4PmxllkY8SnR5KFVJiqxsusc1xv+DZT8tLsW3DsW1H5L5/ACxjEN3G/BNv1QYvRYnCv3xMK+Yhv1ef8eGL6Jp7/hIiGBw5MOOYVD+22Xyw30Q+3E6W+fbwd24c4u7ciO/Am78xMEZ2jXfoyTH9Ed+sJ9pFe9nf2RzHmBHf+w8HND1weNNJp4V+0tKJrqqUeGp6xHiq1QpmDYxqNvmVnS84COQTbG11XVu11exW78jswBe2l+LFu9g5cFRdFU99q86M6La0IwTjgOEkG/eu/uoVktb4dLZZWDFt6w/oRjncGELt56PCmxy1rsq/oeHdvy4YQLT0224HWEW9rDhsm+kSr2YPSH55fts2nLwvyyWtuyDz1oSrWyD34JFqQwtwsoo0tsBouOH7AYKFP3FC+ca+KRkc9Rs6mNdKTyGQqO2AcgczHkdA09lyxUggkXdIL7P7tQGokjhuljSV4HSVt24UliO0zZGlDoawhKgPLdvrBPofp9Mg4Z4QfXy3E60r0ZKOD1ydPr9YwGzcBh27+c3A0GxUn/wKB77jx6MMT8aK/NOZlDMc8NydLfmPmCEaPnbBvR/djQAfTKq9Rhksr3Iw+NKGTv9fM/SNbfasbu2hsV/L/ZUeikQEpAmDH9LlrXMuvKYUfNOY1kFOMHY8xzZAF5TsbTPMe+5rmymkqX38n/AxxRcuMBwAAA'),
}
for name, (expected_sha256, payload) in embedded.items():
    contents = gzip.decompress(base64.b64decode(payload))
    actual_sha256 = hashlib.sha256(contents).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f'Embedded script checksum mismatch for {name}')
    (SCRATCH_DIR / name).write_bytes(contents)
print({name: digest for name, (digest, _) in embedded.items()})


In [ ]:
# Verify the manual parsed labels, then acquire the pinned WHAM release assets.
import hashlib, subprocess, urllib.request

def require_parsed_file(name):
    matches = sorted(PARSED_3DPW_ROOT.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one {name} below {PARSED_3DPW_ROOT}; found {matches}'
        )
    return matches[0]

TRAIN_PARSED = require_parsed_file('3dpw_train_vit.pth')
VAL_PARSED = require_parsed_file('3dpw_val_vit.pth')

WHAM_REPO = SCRATCH_DIR / 'WHAM'
if not (WHAM_REPO / 'lib/models/wham.py').is_file():
    subprocess.run([
        'git', 'clone', '--filter=blob:none', '--no-checkout',
        'https://github.com/yohanshin/WHAM.git', str(WHAM_REPO),
    ], check=True)
    subprocess.run(['git', 'checkout', '--detach', '2b54f7797391c94876848b905ed875b154c4a295'], cwd=WHAM_REPO, check=True)
actual_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=WHAM_REPO, text=True
).strip()
if actual_commit != '2b54f7797391c94876848b905ed875b154c4a295':
    raise RuntimeError(f'Unexpected WHAM commit: {actual_commit}')

WHAM_CHECKPOINT = SCRATCH_DIR / 'wham_vit_bedlam_w_3dpw.pth.tar'
WHAM_CHECKPOINT_URL = (
    'https://huggingface.co/camenduru/WHAM/resolve/main/'
    'wham_vit_bedlam_w_3dpw.pth.tar?download=true'
)
WHAM_CHECKPOINT_SHA256 = (
    '2ba0cb6a7dd597023a6b2ad6056e7a8b6b33144a35fabea570bfd00842cd4eaf'
)
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
if not WHAM_CHECKPOINT.is_file():
    print('Downloading checksum-pinned WHAM release checkpoint...')
    urllib.request.urlretrieve(WHAM_CHECKPOINT_URL, WHAM_CHECKPOINT)
checkpoint_sha256 = sha256_file(WHAM_CHECKPOINT)
if checkpoint_sha256 != WHAM_CHECKPOINT_SHA256:
    raise RuntimeError(f'WHAM checkpoint SHA-256 mismatch: {checkpoint_sha256}')
print({
    'train_parsed': str(TRAIN_PARSED),
    'validation_parsed': str(VAL_PARSED),
    'wham_commit': actual_commit,
    'wham_checkpoint_sha256': checkpoint_sha256,
})


In [ ]:
# Cheap code and data preflight. Training does not start unless both pass.
import subprocess, sys

TRAINER_PATH = SCRATCH_DIR / 'finetune_fastvit_wham_downstream.py'
common = [
    '--work-dir', str(OUTPUT_DIR),
    '--three-dpw-root', str(THREEDPW_ROOT),
    '--sequence-root', str(THREEDPW_ROOT),
    '--train-parsed', str(TRAIN_PARSED),
    '--val-parsed', str(VAL_PARSED),
    '--source-checkpoint', str(SOURCE_CHECKPOINT),
    '--wham-repo', str(WHAM_REPO),
    '--wham-checkpoint', str(WHAM_CHECKPOINT),
    '--clip-length', str(CLIP_LENGTH),
    '--stride', str(STRIDE),
    '--max-clips', str(MAX_CLIPS),
]
subprocess.run([sys.executable, str(TRAINER_PATH), '--self-test', *common], check=True)
subprocess.run([sys.executable, str(TRAINER_PATH), '--inspect-data', *common], check=True)


In [ ]:
# Full downstream-aware run. Expect this cell to take substantial GPU time.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, str(TRAINER_PATH), *common,
    '--batch-size', str(BATCH_SIZE),
    '--workers', str(WORKERS),
    '--head-epochs', str(HEAD_EPOCHS),
    '--last-stage-epochs', str(LAST_STAGE_EPOCHS),
    '--finetune-head-lr', str(FINETUNE_HEAD_LR),
    '--finetune-backbone-lr', str(FINETUNE_BACKBONE_LR),
    '--wham-pose-weight', str(WHAM_POSE_WEIGHT),
    '--wham-root-weight', str(WHAM_ROOT_WEIGHT),
    '--wham-gt-pose-weight', str(WHAM_GT_POSE_WEIGHT),
    '--wham-gt-root-weight', str(WHAM_GT_ROOT_WEIGHT),
    '--rotation-loss', ROTATION_LOSS,
    '--log-every', '50',
]
if CONTINUE_FROM_PHASE3:
    command.append('--stop-when-accepted')
print('Starting frozen-WHAM training; 3DPW test is not an input.')
training_result = subprocess.run(command, check=False)
if training_result.returncode != 0:
    raise RuntimeError(f'Downstream training failed with exit code {training_result.returncode}')


In [ ]:
# Compact result and download links; remove every heavy temporary asset.
import json, shutil
from IPython.display import FileLink, display

REPORT_PATH = OUTPUT_DIR / 'fastvit_hmr2_training_report.json'
CHECKPOINT_PATH = OUTPUT_DIR / 'fastvit_hmr2_best.pth'
HISTORY_PATH = OUTPUT_DIR / 'fastvit_hmr2_history.csv'
for path in (REPORT_PATH, CHECKPOINT_PATH, HISTORY_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Expected output was not produced: {path}')
report = json.loads(REPORT_PATH.read_text())
best = report['best_validation']
compact = {
    'accepted_on_validation': report['accepted_on_validation'],
    'deployment_accepted': report['deployment_accepted'],
    'acceptance_state': report['acceptance_state'],
    'tracks': best['tracks'],
    'frames': best['frames'],
    'gates': best['gates'],
    'feature_cosine_mean': best['feature_cosine_mean'],
    'teacher_pose_error_deg': best['teacher_pose_error_deg'],
    'student_pose_error_deg': best['student_pose_error_deg'],
    'pose_degradation_deg': best['pose_degradation_deg'],
    'relative_pose_degradation': best['relative_pose_degradation'],
    'student_teacher_pose_drift_deg': best['student_teacher_pose_drift_deg'],
    'checkpoint_sha256': report['best_checkpoint_sha256'],
}
print(json.dumps(compact, indent=2))
for path in (CHECKPOINT_PATH, HISTORY_PATH, REPORT_PATH):
    display(FileLink(str(path)))
shutil.rmtree(SCRATCH_DIR, ignore_errors=True)
if report['accepted_on_validation']:
    print('Save a new version, then run the separate untouched-test notebook once.')
else:
    print('Validation rejected this checkpoint. Do not run the untouched test; inspect the report and history.')


## What a successful run means

`accepted_on_validation: true` means the checkpoint passed the predeclared rotation-space substitution gates on 3DPW validation. It intentionally still says `deployment_accepted: false` and `awaiting_untouched_3dpw_test`. Save this notebook version, attach it to the separate test notebook, and run that test exactly once. Do not tune again from its result.
